# Modelamiento
Entrenamiento de modelos.

# 💘 Cupidy — Modelamiento y Evaluación
## Proyecto CRISP-DM | Fase 4: Modelamiento + Evaluación

**Modelos:** Árbol de Decisión, MLP, SVM, KNN, Random Forest, XGBoost, Gradient Boosting  
**Selección estadística:** ANOVA + Tukey HSD  
**Hiperparametrización:** GridSearchCV + BayesSearchCV  
**Evaluación principal:** ROC-AUC en test set (30%)

In [ ]:
import matplotlib
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import joblib
import os
import time

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11
})

PALETTE = {
    "primario": "#FF1493",
    "secundario": "#FFB6C1",
    "negativo": "#FF6B6B"
}

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

print("Librerías cargadas")

## FASE 4 — MODELAMIENTO

### 4.1 Carga de datos preparados

In [ ]:
X_train = pd.read_csv("../data/X_train.csv")
X_test  = pd.read_csv("../data/X_test.csv")

y_train = pd.read_csv("../data/y_train.csv").values.ravel()
y_test  = pd.read_csv("../data/y_test.csv").values.ravel()

with open("../data/feature_names.pkl", "rb") as f:
    feature_names = pickle.load(f)

LEAKAGE_CHECK = [
    "dec","dec_o","like","prob","attr","sinc","intel","fun",
    "amb","shar","attr_o","sinc_o","intel_o","fun_o","amb_o",
    "shar_o","int_corr","met","like_o","prob_o"
]

for col in LEAKAGE_CHECK:
    assert col not in X_train.columns, f"LEAKAGE: {col}"

assert abs(y_train.mean() - 0.5) < 0.02
assert 0.13 < y_test.mean() < 0.22

print("Datos cargados correctamente")
print(X_train.shape, X_test.shape)

### 4.2 Definición de los 7 modelos y función de evaluación

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier

MODELOS = {
    "Arbol": DecisionTreeClassifier(
        random_state=42,
        max_depth=8,
        min_samples_leaf=10
    ),

    "MLP": MLPClassifier(
        random_state=42,
        max_iter=500,
        hidden_layer_sizes=(64,32),
        early_stopping=True
    ),

    "SVM": SVC(
        random_state=42,
        probability=True,
        kernel="rbf",
        C=1.0
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=7,
        metric="euclidean"
    ),

    "RF": RandomForestClassifier(
        random_state=42,
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        n_jobs=-1
    ),

    "XGB": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False,
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        n_jobs=-1
    ),

    "GB": GradientBoostingClassifier(
        random_state=42,
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1
    )
}

def evaluar_en_test(nombre, modelo, X_tr, y_tr, X_te, y_te):

    t0 = time.time()

    modelo.fit(X_tr, y_tr)

    t_fit = time.time() - t0

    y_pred = modelo.predict(X_te)

    idx1 = list(modelo.classes_).index(1)

    y_prob = modelo.predict_proba(X_te)[:, idx1]

    return {
        "Modelo": nombre,
        "Accuracy": round(accuracy_score(y_te, y_pred),4),
        "Precision": round(precision_score(y_te, y_pred),4),
        "Recall": round(recall_score(y_te, y_pred),4),
        "F1": round(f1_score(y_te, y_pred),4),
        "ROC_AUC": round(roc_auc_score(y_te, y_prob),4),
        "Tiempo_s": round(t_fit,2),
        "_modelo": modelo,
        "_yprob": y_prob,
        "_ypred": y_pred
    }

print("Modelos configurados")

### 4.3 Cross-Validation 10-fold + evaluación en test

In [ ]:
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

cv_scores = {}
test_results = []

for nombre, modelo in MODELOS.items():

    scores = cross_val_score(
        modelo,
        X_train,
        y_train,
        cv=skf,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_scores[nombre] = scores

    res = evaluar_en_test(
        nombre,
        modelo,
        X_train,
        y_train,
        X_test,
        y_test
    )

    res["CV_mean"] = round(scores.mean(),4)
    res["CV_std"] = round(scores.std(),4)

    test_results.append(res)

    print(
        f"{nombre} | "
        f"CV={scores.mean():.4f} | "
        f"TEST={res['ROC_AUC']:.4f}"
    )

df_resultados = pd.DataFrame(test_results).sort_values(
    "ROC_AUC",
    ascending=False
)

df_resultados

### 4.4 Visualizaciones comparativas

In [ ]:
os.makedirs("../reports", exist_ok=True)

metricas_plot = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

fig, axes = plt.subplots(1,5, figsize=(24,5))

palette = sns.color_palette("RdPu", len(MODELOS))

for i, met in enumerate(metricas_plot):

    nombres = df_resultados["Modelo"].values
    valores = df_resultados[met].values

    axes[i].bar(
        nombres,
        valores,
        color=palette
    )

    axes[i].set_title(met)
    axes[i].set_ylim(0,1.1)

plt.tight_layout()

plt.savefig(
    "../reports/comparacion_metricas.png",
    dpi=150
)

plt.show()

### 4.5 Selección estadística: ANOVA + Tukey HSD

In [ ]:
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

grupos = [cv_scores[n] for n in cv_scores.keys()]

f_stat, p_valor = stats.f_oneway(*grupos)

print(f"F={f_stat:.4f}")
print(f"p={p_valor:.6f}")

all_scores = np.concatenate(grupos)

all_names = np.repeat(
    list(cv_scores.keys()),
    [len(g) for g in grupos]
)

tukey = pairwise_tukeyhsd(
    all_scores,
    all_names,
    alpha=0.05
)

print(tukey)

### 4.6 Hiperparametrización — GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

GRIDS = {
    "RF": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators":[100,200],
            "max_depth":[10,15,None]
        }
    },

    "XGB": {
        "model": XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            random_state=42
        ),

        "params": {
            "n_estimators":[100,200],
            "max_depth":[3,5,7]
        }
    }
}

top3 = df_resultados.head(3)

top3_nombres = top3["Modelo"].tolist()

skf5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

mejores_grid = {}

for nombre in top3_nombres:

    if nombre not in GRIDS:
        continue

    config = GRIDS[nombre]

    gs = GridSearchCV(
        config["model"],
        config["params"],
        cv=skf5,
        scoring="roc_auc",
        n_jobs=-1
    )

    gs.fit(X_train, y_train)

    mejores_grid[nombre] = gs.best_estimator_

    print(nombre)
    print(gs.best_score_)
    print(gs.best_params_)

### 4.7 Hiperparametrización — BayesSearchCV

In [ ]:
from skopt import BayesSearchCV
from skopt.space import Integer, Real

cfg = {
    "model": RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),

    "space": {
        "n_estimators": Integer(100,500),
        "max_depth": Integer(5,30),
        "min_samples_leaf": Integer(2,20)
    }
}

bsv = BayesSearchCV(
    cfg["model"],
    cfg["space"],
    n_iter=30,
    cv=skf5,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1
)

bsv.fit(X_train, y_train)

modelo_bayes = bsv.best_estimator_

print(bsv.best_score_)
print(bsv.best_params_)

## FASE 5 — EVALUACIÓN FINAL

In [ ]:
res = evaluar_en_test(
    "Modelo_Final",
    modelo_bayes,
    X_train,
    y_train,
    X_test,
    y_test
)

modelo_final = res["_modelo"]

y_prob_final = res["_yprob"]

y_pred_final = res["_ypred"]

auc_final = res["ROC_AUC"]

print(res)

cm = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="RdPu"
)

plt.title("Matriz de Confusión")

plt.show()

fpr, tpr, _ = roc_curve(y_test, y_prob_final)

plt.figure(figsize=(7,6))

plt.plot(
    fpr,
    tpr,
    label=f"AUC={auc_final:.3f}"
)

plt.plot([0,1],[0,1],"k--")

plt.legend()

plt.title("Curva ROC")

plt.show()

print(classification_report(y_test, y_pred_final))

### 4.9 Feature Importance + SHAP

In [ ]:
if hasattr(modelo_final, "feature_importances_"):

    imp = pd.DataFrame({
        "Feature": feature_names,
        "Importancia": modelo_final.feature_importances_
    })

    imp = imp.sort_values(
        "Importancia",
        ascending=False
    )

    top15 = imp.head(15)

    plt.figure(figsize=(10,7))

    plt.barh(
        top15["Feature"],
        top15["Importancia"],
        color=PALETTE["primario"]
    )

    plt.gca().invert_yaxis()

    plt.title("Top 15 Variables")

    plt.tight_layout()

    plt.savefig(
        "../reports/feature_importance.png",
        dpi=150
    )

    plt.show()

try:
    import shap

    explainer = shap.TreeExplainer(modelo_final)

    shap_vals = explainer.shap_values(X_test[:300])

    shap.summary_plot(
        shap_vals,
        X_test[:300]
    )

except Exception as e:
    print(e)

### 4.10 Guardar modelo final y métricas

In [ ]:
os.makedirs("../models", exist_ok=True)

joblib.dump(
    modelo_final,
    "../models/pipeline_match_predictor.pkl"
)

idx_clase1 = list(modelo_final.classes_).index(1)

metricas_finales = {
    "modelo": "Modelo_Final",
    "accuracy": float(res["Accuracy"]),
    "precision": float(res["Precision"]),
    "recall": float(res["Recall"]),
    "f1": float(res["F1"]),
    "roc_auc": float(res["ROC_AUC"]),
    "n_features": len(feature_names),
    "feature_names": list(feature_names),
    "idx_clase1": idx_clase1
}

with open("../models/metricas_finales.pkl", "wb") as f:
    pickle.dump(metricas_finales, f)

print("Modelo guardado")
print("Métricas guardadas")